# 🧬 PhysioTwin Dual-AI Google Colab Training Notebook
This notebook runs Phase 1 of the training pipeline on Google Colab GPU (T4 / A100).
It trains both the Clinical LLM (Unsloth QLoRA SFT+DPO) and the Live Webcam Pose Model (YOLO Pose), quantizes to GGUF, and uploads all models directly to Hugging Face Hub.

### Step 1: Verify GPU Access

In [ ]:
!nvidia-smi

### Step 2: Install High-Performance Unsloth & Vision Stack

In [ ]:
# Install Unsloth and required fine-tuning dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets
!pip install ultralytics opencv-python onnx onnxruntime huggingface_hub

### Step 3: Clone / Clean Codebase & Prepare Datasets

In [ ]:
import os, shutil
%cd /content

# If regi-twin folder exists, remove it cleanly before re-cloning
if os.path.exists('/content/regi-twin'):
    shutil.rmtree('/content/regi-twin')

# Clone repository
!git clone https://github.com/kamaleshramesh7275-cloud/regi-twin.git /content/regi-twin

# Change directory to ai_pipeline using absolute path
%cd /content/regi-twin/backend/ai_pipeline
!python prepare_dataset.py

### Step 4: Run 2-Stage Unsloth LLM Training (SFT + DPO)

In [ ]:
%cd /content/regi-twin/backend/ai_pipeline
!python train_model.py

### Step 5: Train Live Webcam Pose Detection Model & Export ONNX Web

In [ ]:
%cd /content/regi-twin/backend/scripts
!python extract_video_frames.py
!python auto_annotate_pose.py
!python train_pose_model.py
!python export_onnx_web.py

### Step 6: Quantize LLM to GGUF and Upload Models to Hugging Face Hub

In [ ]:
%cd /content/regi-twin/backend/ai_pipeline
import os
HF_TOKEN = "hf_YOUR_HUGGINGFACE_WRITE_TOKEN" # 👈 Replace with your HF write token
REPO_ID = "YOUR_USERNAME/physiotwin-gguf"      # 👈 Replace with your HF repo ID

!python quantize_and_upload.py --repo_id {REPO_ID} --hf_token {HF_TOKEN}

🎉 **Training & Cloud Upload Complete!**
You can now close Colab and run Phase 2 on your local machine using Ollama and the browser ONNX model.